In [ ]:
import pandas as pd

In [ ]:
race_data = pd.read_csv('../data/lap_weather_data_2018_2025.csv')

In [ ]:
# Rank races by the share of laps marked as rainy
rainy_races = race_data.groupby(['Year', 'EventName'])['Rainfall'].transform('any')

race_rain_pct = (
    race_data[rainy_races]
    .groupby(['Year', 'EventName'])['Rainfall']
    .mean()
    .mul(100)
    .round(2)
    .loc[lambda s: s > 10]
    .reset_index(name='PercentRain')
    .sort_values('PercentRain', ascending=False)
    .reset_index(drop=True)
)

race_rain_pct

In [ ]:
race_data['LapTime'] = pd.to_timedelta(race_data['LapTime']).dt.total_seconds()

In [ ]:
# Compare each 2024 British GP lap to that lap's field median
median_laps = (
    race_data.loc[
        (race_data['Year'] == 2024)
        & (race_data['EventName'] == 'British Grand Prix')
        & (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
    ]
    .groupby('LapNumber', as_index=False)['LapTime']
    .median()
    .rename(columns={'LapTime': 'MedianLapTime'})
)

driver_laps = (
    race_data.loc[
        (race_data['Year'] == 2024)
        & (race_data['EventName'] == 'British Grand Prix')
        & (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
        & (race_data['LapNumber'].isin(median_laps['LapNumber']))
    ][['Driver', 'LapNumber', 'LapTime', 'Rainfall']]
    .reset_index(drop=True)
)

driver_laps_diff = driver_laps.merge(
    median_laps,
    on='LapNumber',
    how='left'
)

driver_laps_diff['LapTimeDiff'] = (
    driver_laps_diff['LapTime'] 
    - driver_laps_diff['MedianLapTime']
)

In [ ]:
# Measure which drivers improved most relative to the field in rain
rain_performance = (
    driver_laps_diff.loc[driver_laps_diff['Rainfall'] == True]
    .groupby('Driver')['LapTimeDiff']
    .mean()
    .reset_index(name='AvgRainDiff')
)

dry_performance = (
    driver_laps_diff.loc[driver_laps_diff['Rainfall'] == False]
    .groupby('Driver')['LapTimeDiff']
    .mean()
    .reset_index(name='AvgDryDiff')
)

driver_performance = rain_performance.merge(
    dry_performance,
    on='Driver'
)

driver_performance['AvgGain'] = (
    driver_performance['AvgDryDiff']
    - driver_performance['AvgRainDiff']
)

driver_performance.sort_values('WeatherGain', ascending=False).reset_index(drop=True)